# Datenaufbereitung und Merge der CSV-Dateien

Dieses Notebook bereitet die vier Datensätze **HDR**, **PWT**, **EFOTW** und **WHR** auf, speichert sie jeweils als CSV-Datei und merged sie anschließend über die Spalte `country`.

Zeitraum: **2010 bis 2023**

## 0. Imports und Grundeinstellungen

Hier werden alle benötigten Pakete geladen. Für `.ods`-Dateien wird `odfpy` benötigt.

In [3]:
import pandas as pd
from functools import reduce

## 0.1. HDR-Daten aufbereiten → `hdr.csv`

Die HDR-Datei liegt als `.ods` vor. Die ersten Zeilen enthalten Überschriften/Formatierung, deshalb werden sie mit `skiprows=4` übersprungen. Danach werden nur die relevanten Spalten behalten.

In [4]:
# HDR-Datei laden
HDR = pd.read_excel(
    r"Data\HDR25_Statistical_Annex_HDI_Trends_Table.ods",
    engine="odf",
    skiprows=4
)

# Leere Zeilen ohne HDI-Rang entfernen
HDR = HDR.dropna(subset=["HDI rank"])

# Komplett leere Spalten entfernen
HDR = HDR.dropna(axis=1, how="all")

# Nur relevante Spalten behalten
HDR = HDR[["Country", "HDI rank", "2010-2023"]]

# Vorschau
print(HDR.head())

# Als CSV speichern
HDR.to_csv("Datapreps/hdr.csv", index=False)

       Country  HDI rank 2010-2023
1      Iceland       1.0  0.298979
2       Norway       2.0  0.184763
3  Switzerland       2.0  0.201057
4      Denmark       4.0  0.343981
5      Germany       5.0   0.18691


## 0.2. PWT-Daten aufbereiten → `PWT_mean.csv`

Die PWT-Daten werden auf die Jahre **2010 bis 2023** gefiltert. Anschließend wird pro Land der Durchschnitt über alle numerischen Spalten berechnet.

In [5]:
# PWT-Datei laden
PWT = pd.read_excel(r"Data\pwt110.xlsx")

# Nur Jahre 2010 bis 2023 behalten
PWT = PWT[PWT["year"].between(2010, 2023)]

# Nur relevante Spalten behalten
PWT = PWT[[
    "countrycode",
    "country",
    "year",
    "rgdpo"
]]


# Wichtig: nach Land + Jahr sortieren
PWT = PWT.sort_values(["country", "year"])

# Wachstumsrate der einzelnen Länder berechnen
PWT["rgdpo growth"] = (
    PWT.groupby("country")["rgdpo"]
    .pct_change()
)

# Fehler / NaN ersetzen
PWT["rgdpo growth"] = (
    PWT["rgdpo growth"]
    .fillna(0)
)

# Standardabweichung des rgdpo-Wachstums pro Land berechnen
PWT_growth_std = (
    PWT.groupby("country", as_index=False)["rgdpo growth"]
    .std()
    .rename(columns={"rgdpo growth": "rgdpo growth std"})
)

# Durchschnitt pro Land berechnen
PWT_mean = PWT.groupby(
    "country",
    as_index=False
).mean(numeric_only=True)

# Standardabweichung hinzufügen
PWT_mean = PWT_mean.merge(
    PWT_growth_std,
    on="country",
    how="left"
)

# Zeitraum setzen
PWT_mean["year"] = "2010-2023"

# Vorschau
print(PWT_mean.head())

# Als CSV speichern
PWT_mean.to_csv("Datapreps/PWT_mean.csv", index=False)

               country       year          rgdpo  rgdpo growth  \
0              Albania  2010-2023   37933.725725      0.024395   
1              Algeria  2010-2023  606469.843750      0.009825   
2               Angola  2010-2023  238042.943080      0.020677   
3             Anguilla  2010-2023     302.352133      0.018308   
4  Antigua and Barbuda  2010-2023    1994.485447      0.037070   

   rgdpo growth std  
0          0.041700  
1          0.043242  
2          0.094117  
3          0.362251  
4          0.244111  


## 0.3. EFOTW-Daten aufbereiten → `EFOTW_mean.csv`

Die EFOTW-Daten werden auf **2010 bis 2023** gefiltert. Danach wird pro Land der Durchschnitt berechnet.

In [6]:
# EFOTW-Datei laden
EFOTW = pd.read_excel(r"Data\efotw-2025-master-index-data-for-researchers-iso.xlsx")

# Spaltennamen bereinigen
EFOTW.columns = EFOTW.columns.str.strip()

income_mapping = {
    "Low Income": 1,
    "Lower-Middle Income": 2,
    "Upper-Middle Income": 3,
    "High Income": 4
}

EFOTW["World Bank Current Income Classification, 1990-Present"] = (
    EFOTW["World Bank Current Income Classification, 1990-Present"]
    .map(income_mapping)
)

# Nur Jahre 2010 bis 2023 behalten
EFOTW = EFOTW[EFOTW["Year"].between(2010, 2023)]

# Nur relevante Spalten behalten
EFOTW = EFOTW[["ISO_Code","Countries", "Year", "Summary", "Standard Deviation of the 5 EFW Areas", "World Bank Current Income Classification, 1990-Present"]] 

# Durchschnitt pro Land berechnen
EFOTW_mean = EFOTW.groupby("Countries", as_index=False).mean(numeric_only=True)

# Jahr/Zeitraum einheitlich setzen
EFOTW_mean["Year"] = "2010-2023"

# Vorschau
print(EFOTW_mean.head())

# Als CSV speichern
EFOTW_mean.to_csv("Datapreps/EFOTW_mean.csv", index=False)

   Countries       Year   Summary  Standard Deviation of the 5 EFW Areas  \
0    Albania  2010-2023  7.584226                               1.493865   
1    Algeria  2010-2023  4.665171                               1.473538   
2     Angola  2010-2023  4.854164                               1.545606   
3  Argentina  2010-2023  5.133541                               1.049611   
4    Armenia  2010-2023  7.314481                               1.404578   

   World Bank Current Income Classification, 1990-Present  
0                                           2.928571       
1                                           2.785714       
2                                           2.357143       
3                                           3.142857       
4                                           2.500000       


## 0.4. WHR-Daten aufbereiten → `WHR_mean.csv`

Die WHR-Daten werden ebenfalls auf **2010 bis 2023** gefiltert. Danach wird pro Land der Durchschnitt berechnet.

In [7]:
# WHR-Datei laden
WHR = pd.read_excel(r"Data\WHR26_Data_Figure_2.1.xlsx")

# Spaltennamen bereinigen
WHR.columns = WHR.columns.str.strip()

# Nur relevante Spalten behalten
WHR = WHR[["Year","Country name", "Life evaluation (3-year average)"]]

# Nur Jahre 2010 bis 2023 behalten
WHR = WHR[WHR["Year"].between(2010, 2023)]

# Durchschnitt pro Land berechnen
WHR_mean = WHR.groupby("Country name", as_index=False).mean(numeric_only=True)

# Jahr/Zeitraum einheitlich setzen
WHR_mean["Year"] = "2010-2023"

# Vorschau
print(WHR_mean.head())

# Als CSV speichern
WHR_mean.to_csv("Datapreps/WHR_mean.csv", index=False)

  Country name       Year  Life evaluation (3-year average)
0  Afghanistan  2010-2023                          3.077992
1      Albania  2010-2023                          5.002225
2      Algeria  2010-2023                          5.402925
3       Angola  2010-2023                          4.209333
4    Argentina  2010-2023                          6.265308


## 1.1. CSV-Dateien laden und Spaltennamen vereinheitlichen

Jetzt werden die vier erzeugten CSV-Dateien geladen. Damit der Merge funktioniert, werden alle Spaltennamen klein geschrieben und Leerzeichen am Anfang/Ende entfernt.

In [8]:
# Aufbereitete CSV-Dateien laden
efotw = pd.read_csv("Datapreps/EFOTW_mean.csv")
hdr   = pd.read_csv("Datapreps/hdr.csv")
pwt   = pd.read_csv("Datapreps/PWT_mean.csv")
whr   = pd.read_csv("Datapreps/WHR_mean.csv")

# Spaltennamen vereinheitlichen
for df in [efotw, hdr, pwt, whr]:
    df.columns = df.columns.str.strip().str.lower()

# Länder-Spalten einheitlich in "country" umbenennen
efotw = efotw.rename(columns={"countries": "country"})
whr   = whr.rename(columns={"country name": "country"})

# Kontrolle der Spaltennamen
print(efotw.columns.tolist())
print(hdr.columns.tolist())
print(pwt.columns.tolist())
print(whr.columns.tolist())

['country', 'year', 'summary', 'standard deviation of the 5 efw areas', 'world bank current income classification, 1990-present']
['country', 'hdi rank', '2010-2023']
['country', 'year', 'rgdpo', 'rgdpo growth', 'rgdpo growth std']
['country', 'year', 'life evaluation (3-year average)']


## 1.2. Nicht benötigte `year`-Spalten entfernen

Da alle Datensätze bereits auf den Zeitraum **2010 bis 2023** aggregiert wurden, wird `year` aus den gemittelten Datensätzen entfernt.

In [9]:
efotw = efotw.drop(columns="year", errors="ignore")
pwt   = pwt.drop(columns="year", errors="ignore")
whr   = whr.drop(columns="year", errors="ignore")

# Kontrolle der Spaltennamen
print(efotw.columns.tolist())
print(hdr.columns.tolist())
print(pwt.columns.tolist())
print(whr.columns.tolist())

['country', 'summary', 'standard deviation of the 5 efw areas', 'world bank current income classification, 1990-present']
['country', 'hdi rank', '2010-2023']
['country', 'rgdpo', 'rgdpo growth', 'rgdpo growth std']
['country', 'life evaluation (3-year average)']


## 1.3. Datensätze über `country` mergen

Mit `how="inner"` bleiben nur Länder erhalten, die in allen vier Datensätzen vorkommen.

In [10]:
dfs = [efotw, hdr, pwt, whr]

merged = reduce(
    lambda left, right: pd.merge(left, right, on="country", how="inner"),
    dfs
)
merged = merged.drop(columns="rgdpo growth", errors="ignore")

print(merged.head())
print(merged.shape)

     country   summary  standard deviation of the 5 efw areas  \
0    Albania  7.584226                               1.493865   
1    Algeria  4.665171                               1.473538   
2     Angola  4.854164                               1.545606   
3  Argentina  5.133541                               1.049611   
4    Armenia  7.314481                               1.404578   

   world bank current income classification, 1990-present  hdi rank  \
0                                           2.928571           71.0   
1                                           2.785714           96.0   
2                                           2.357143          148.0   
3                                           3.142857           47.0   
4                                           2.500000           69.0   

            2010-2023         rgdpo  rgdpo growth std  \
0  0.4003629980676049  3.793373e+04          0.041700   
1  0.4686985271329158  6.064698e+05          0.043242   
2  1.192832

## 1.4. Spaltennamen anpassen, Zeilen entfernen und CSV speichern

Die Spalte `2010-2023` aus HDR wird sprechender benannt. Zeilen, welche nicht in jeder Spalte einen Wert enthalten, werden gelöscht und danach wird die finale Datei als `merged.csv` gespeichert.

In [11]:
merged = merged.rename(columns={
    "country": "country",
    "summary":"efotw_summary", 
    "area 1":"efotw_area 1",
    "area 2":"efotw_area 2",
    "area 3":"efotw_area 3",
    "area 4":"efotw_area 4",
    "area 5":"efotw_area 5",
    "standard deviation of the 5 efw areas":"efotw_standard_deviation_of_the_5_efw_areas",
    "world bank current income classification, 1990-present":"efotw_world_bank_current_income_classification",
    "year": "year",
    "2010-2023":"Average_annual_HDI_growth_2010-2023",
    "hdi rank":"hdi_rank",
    "rgdpo per capita":"rgdpo_per_capita",
    "rgdpo growth std":"rgdpo_growth_std",
    "life evaluation (3-year average)":"life_evaluation_(3-year_average)"
})

merged = merged.dropna()
print(merged.head())
merged.to_csv("merged.csv", index=False)

     country  efotw_summary  efotw_standard_deviation_of_the_5_efw_areas  \
0    Albania       7.584226                                     1.493865   
1    Algeria       4.665171                                     1.473538   
2     Angola       4.854164                                     1.545606   
3  Argentina       5.133541                                     1.049611   
4    Armenia       7.314481                                     1.404578   

   efotw_world_bank_current_income_classification  hdi_rank  \
0                                        2.928571      71.0   
1                                        2.785714      96.0   
2                                        2.357143     148.0   
3                                        3.142857      47.0   
4                                        2.500000      69.0   

  Average_annual_HDI_growth_2010-2023         rgdpo  rgdpo_growth_std  \
0                  0.4003629980676049  3.793373e+04          0.041700   
1                 

In [14]:
variable_dictionary = pd.DataFrame([
    {"Variable": "country", "Role": "Identifier", "Interpretation": "Country name; excluded from model features."},
    {"Variable": "rgdpo_growth_std", "Role": "Target", "Interpretation": "Volatility of real output growth; higher values indicate lower macroeconomic stability."},
    {"Variable": "efotw_summary", "Role": "Institutional predictor", "Interpretation": "Overall institutional/economic freedom score."},
    {"Variable": "efotw_standard_deviation_of_the_5_efw_areas", "Role": "Institutional predictor", "Interpretation": "Dispersion across institutional/economic freedom areas."},
    {"Variable": "efotw_world_bank_current_income_classification", "Role": "Welfare/development predictor", "Interpretation": "World Bank income classification encoded numerically in the dataset."},
    {"Variable": "hdi_rank", "Role": "Development predictor", "Interpretation": "Human Development Index rank; lower rank means higher development."},
    {"Variable": "Average_annual_HDI_growth_2010-2023", "Role": "Development predictor", "Interpretation": "Average annual HDI growth over 2010–2023."},
    {"Variable": "rgdpo", "Role": "Scale/economic size predictor", "Interpretation": "Real GDP/output level; transformed to log scale in feature engineering."},
    {"Variable": "life_evaluation_(3-year_average)", "Role": "Subjective well-being predictor", "Interpretation": "Three-year average life evaluation score."},
])

display(variable_dictionary)

,Variable,Role,Interpretation
0,country,Identifier,Country name; excluded from model features.
1,rgdpo_growth_std,Target,Volatility of real output growth; higher value...
2,efotw_summary,Institutional predictor,Overall institutional/economic freedom score.
3,efotw_standard_deviation_of_the_5_efw_areas,Institutional predictor,Dispersion across institutional/economic freed...
4,efotw_world_bank_current_income_classification,Welfare/development predictor,World Bank income classification encoded numer...
5,hdi_rank,Development predictor,Human Development Index rank; lower rank means...
6,Average_annual_HDI_growth_2010-2023,Development predictor,Average annual HDI growth over 2010–2023.
7,rgdpo,Scale/economic size predictor,Real GDP/output level; transformed to log scal...
8,life_evaluation_(3-year_average),Subjective well-being predictor,Three-year average life evaluation score.


## 3. Feature engineering

### Siehe PWT einleseprozess